# Create global facet plots of annual runoff onset and accomapnying temporal resolution

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import rioxarray as rxr
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import numpy as np
import zarr
from dask.distributed import Client, LocalCluster
from matplotlib.lines import Line2D

In [ ]:
# cluster = LocalCluster(
#     n_workers=8,
#     threads_per_worker=4,
#     memory_limit='15GB',       # changed from 10 to 15GB
#     local_directory='/tmp/dask-spill',
# )
# client = Client(cluster)

# zarr.config.set({
#     'async.concurrency': 128,   # 32 to 128GB
#     'threading.max_workers': 16, # codec threads = CPU count
# })

# client

In [ ]:
config = Config('config/global_config_v9.txt')

In [ ]:
coarsen_factor = 20
store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_{config.version}_coarsened_{coarsen_factor}_ds.zarr")
global_coarsened_ds = xr.open_zarr(store, consolidated=True, decode_coords='all',chunks="auto")#{'latitude':3*2048, 'longitude':3*2048}
global_coarsened_ds = global_coarsened_ds.rio.write_crs('EPSG:4326')
global_coarsened_ds

In [ ]:
global_coarsened_ds = global_coarsened_ds.coarsen(latitude=6, longitude=6, boundary='trim').mean().compute()
global_coarsened_ds

In [ ]:
global_hillshade_robinson_da = rxr.open_rasterio('../data/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().coarsen(x=15, y=15, boundary='trim').mean().compute()
global_hillshade_robinson_da

In [ ]:
# fig, ax = plt.subplots(figsize=(12, 7), subplot_kw=dict(projection=ccrs.Robinson()))
# global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), add_colorbar=False)

In [ ]:
# # First figure: 2015-2019
# f1, axs1 = plt.subplots(nrows=5, ncols=2, figsize=(6,8), 
#                         subplot_kw=dict(projection=ccrs.Robinson()),
#                         dpi=300, layout='constrained')


# dowy_cbar_ticks = [110, 150, 190, 230, 270]
# days_cbar_ticks = [1, 5, 10, 15, 20]

# for water_year in global_coarsened_ds['water_year'].values[:5]:  # 2015-2019
#     print(water_year)
    
#     row_idx = int(water_year) - 2015
    
#     if water_year == 2019:
#         colorbar = True
#         cbar_dowy = {'label':'DOWY','orientation':'horizontal', 'ticks': dowy_cbar_ticks}
#         cbar_days = {'label':'Days','orientation':'horizontal', 'ticks': days_cbar_ticks}
#     else:
#         colorbar = False
#         cbar_dowy = None
#         cbar_days = None
    
#     # Runoff onset
#     ax = axs1[row_idx, 0]
#     global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', 
#                                              transform=ccrs.Robinson(), 
#                                              add_colorbar=False)
    
#     global_coarsened_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(
#         ax=ax,
#         cmap='viridis',
#         vmin=110,
#         vmax=270,
#         transform=ccrs.PlateCarree(),
#         add_colorbar=colorbar,
#         cbar_kwargs=cbar_dowy
#     )
#     ax.set_title('')
    
#     # Add row labels using text
#     ax.text(0, 0.5, f'WY{int(water_year)}', transform=ax.transAxes,
#             fontsize=10, rotation=90,fontweight='bold', va='center', ha='right')
    
#     # Temporal resolution
#     ax = axs1[row_idx, 1]
#     global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', 
#                                              transform=ccrs.Robinson(), 
#                                              add_colorbar=False)
    
#     global_coarsened_ds["temporal_resolution"].sel(water_year=water_year).plot.imshow(
#         ax=ax,
#         cmap="summer",
#         vmin=1,
#         vmax=20,
#         transform=ccrs.PlateCarree(),
#         add_colorbar=colorbar,
#         cbar_kwargs=cbar_days
#     )
#     ax.set_title('')

# # Add column titles
# axs1[0, 0].set_title('Snowmelt runoff onset', fontsize=11, fontweight='bold', pad=10)
# axs1[0, 1].set_title('Temporal resolution', fontsize=11, fontweight='bold', pad=10)

# f1.savefig('figures/global_annual_runoff_onset_and_temporal_res_with_hillshade_2015_2019.png', 
#            dpi=300, bbox_inches='tight')

# # Second figure: 2020-2024
# f2, axs2 = plt.subplots(nrows=5, ncols=2, figsize=(6,8), 
#                         subplot_kw=dict(projection=ccrs.Robinson()),
#                         dpi=300, layout='constrained')

# for water_year in global_coarsened_ds['water_year'].values[5:]:  # 2020-2024
#     print(water_year)
    
#     row_idx = int(water_year) - 2020
    
#     if water_year == 2024:
#         colorbar = True
#         cbar_dowy = {'label':'DOWY','orientation':'horizontal', 'ticks': dowy_cbar_ticks}
#         cbar_days = {'label':'Days','orientation':'horizontal', 'ticks': days_cbar_ticks}
#     else:
#         colorbar = False
#         cbar_dowy = None
#         cbar_days = None
    
#     # Runoff onset
#     ax = axs2[row_idx, 0]
#     global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', 
#                                              transform=ccrs.Robinson(), 
#                                              add_colorbar=False)
    
#     global_coarsened_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(
#         ax=ax,
#         cmap='viridis',
#         vmin=110,
#         vmax=270,
#         transform=ccrs.PlateCarree(),
#         add_colorbar=colorbar,
#         cbar_kwargs=cbar_dowy
#     )
#     ax.set_title('')
    
#     # Add row labels using text
#     ax.text(-0.0, 0.5, f'WY{int(water_year)}', transform=ax.transAxes,
#             fontsize=10, fontweight='bold', rotation=90, va='center', ha='right')
    
#     # Temporal resolution
#     ax = axs2[row_idx, 1]
#     global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', 
#                                              transform=ccrs.Robinson(), 
#                                              add_colorbar=False)
    
#     global_coarsened_ds["temporal_resolution"].sel(water_year=water_year).plot.imshow(
#         ax=ax,
#         cmap="summer",
#         vmin=1,
#         vmax=20,
#         transform=ccrs.PlateCarree(),
#         add_colorbar=colorbar,
#         cbar_kwargs=cbar_days
#     )
#     ax.set_title('')

# # Add column titles
# axs2[0, 0].set_title('Snowmelt runoff onset', fontsize=11, fontweight='bold', pad=10)
# axs2[0, 1].set_title('Temporal resolution', fontsize=11, fontweight='bold', pad=10)

# f2.savefig('figures/global_annual_runoff_onset_and_temporal_res_with_hillshade_2020_2024.png', 
#            dpi=300, bbox_inches='tight')

In [ ]:
dowy_cbar_ticks = [110, 150, 190, 230, 270]
days_cbar_ticks = [1, 5, 10, 15, 20]

# Top-level figure with two subfigures (left: WY2015–2019, right: WY2020–2024).
# Each subfigure runs constrained layout independently, so colorbars in one
# group don't bleed into the spacing of the other.
f = plt.figure(figsize=(12, 8), dpi=300, layout='constrained')
subfigs = f.subfigures(1, 2, wspace=0.06)  # gap between the two groups

axs_left  = subfigs[0].subplots(nrows=5, ncols=2,
                                 subplot_kw=dict(projection=ccrs.Robinson()))
axs_right = subfigs[1].subplots(nrows=5, ncols=2,
                                 subplot_kw=dict(projection=ccrs.Robinson()))

# ── Left block: WY 2015–2019 ──────────────────────────────────────────────────
for water_year in global_coarsened_ds['water_year'].values[:5]:
    row_idx  = int(water_year) - 2015
    colorbar = water_year == 2019
    cbar_dowy = {'label': 'DOWY', 'orientation': 'horizontal', 'ticks': dowy_cbar_ticks} if colorbar else None
    cbar_days = {'label': 'days', 'orientation': 'horizontal', 'ticks': days_cbar_ticks} if colorbar else None

    ax = axs_left[row_idx, 0]
    global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), add_colorbar=False)
    global_coarsened_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(
        ax=ax, cmap='viridis', vmin=110, vmax=270,
        transform=ccrs.PlateCarree(), add_colorbar=colorbar, cbar_kwargs=cbar_dowy)
    ax.set_title('')
    ax.text(-0.03, 0.5, f'WY{int(water_year)}', transform=ax.transAxes,
            fontsize=10, rotation=90, fontweight='bold', va='center', ha='right')

    ax = axs_left[row_idx, 1]
    global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), add_colorbar=False)
    global_coarsened_ds['temporal_resolution'].sel(water_year=water_year).plot.imshow(
        ax=ax, cmap='summer', vmin=1, vmax=20,
        transform=ccrs.PlateCarree(), add_colorbar=colorbar, cbar_kwargs=cbar_days)
    ax.set_title('')

# ── Right block: WY 2020–2024 ─────────────────────────────────────────────────
for water_year in global_coarsened_ds['water_year'].values[5:]:
    row_idx  = int(water_year) - 2020
    colorbar = water_year == 2024
    cbar_dowy = {'label': 'DOWY', 'orientation': 'horizontal', 'ticks': dowy_cbar_ticks} if colorbar else None
    cbar_days = {'label': 'days', 'orientation': 'horizontal', 'ticks': days_cbar_ticks} if colorbar else None

    ax = axs_right[row_idx, 0]
    global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), add_colorbar=False)
    global_coarsened_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(
        ax=ax, cmap='viridis', vmin=110, vmax=270,
        transform=ccrs.PlateCarree(), add_colorbar=colorbar, cbar_kwargs=cbar_dowy)
    ax.set_title('')
    ax.text(-0.03, 0.5, f'WY{int(water_year)}', transform=ax.transAxes,
            fontsize=10, rotation=90, fontweight='bold', va='center', ha='right')

    ax = axs_right[row_idx, 1]
    global_hillshade_robinson_da.plot.imshow(ax=ax, cmap='gray', transform=ccrs.Robinson(), add_colorbar=False)
    global_coarsened_ds['temporal_resolution'].sel(water_year=water_year).plot.imshow(
        ax=ax, cmap='summer', vmin=1, vmax=20,
        transform=ccrs.PlateCarree(), add_colorbar=colorbar, cbar_kwargs=cbar_days)
    ax.set_title('')

# ── Column titles ─────────────────────────────────────────────────────────────
axs_left[0,  0].set_title('Snowmelt runoff onset',  fontsize=11, fontweight='bold', pad=4)
axs_left[0,  1].set_title('Temporal resolution',    fontsize=11, fontweight='bold', pad=4)
axs_right[0, 0].set_title('Snowmelt runoff onset',  fontsize=11, fontweight='bold', pad=4)
axs_right[0, 1].set_title('Temporal resolution',    fontsize=11, fontweight='bold', pad=4)

# ── Vertical separator between the two subfigures ────────────────────────────
# Force layout so subfigure bounding boxes are finalised before we read them.
f.canvas.draw()

# get_window_extent() returns pixel coords; convert to figure (0–1) coords.
inv = f.transFigure.inverted()
bbox_l = inv.transform(subfigs[0].get_window_extent())
bbox_r = inv.transform(subfigs[1].get_window_extent())
x_sep  = (bbox_l[1, 0] + bbox_r[0, 0]) / 2   # midpoint between right edge of left & left edge of right

f.add_artist(
    Line2D([x_sep, x_sep], [0.02, 0.98],
           transform=f.transFigure,
           color='lightgrey', linewidth=1.5, zorder=10)
)

f.savefig('figures/global_annual_runoff_onset_and_temporal_res_with_hillshade_2015_2024.png',
          dpi=300, bbox_inches='tight')